# Unified Bombcell Runner (Grant)

Single entrypoint replacing batch, single-probe, and NP2.0 rerun notebooks.

#### General pip install for loading .env

In [ ]:
#!pip install python-dotenv

#### Imports

In [1]:
from pathlib import Path
import re
from typing import Any, Dict
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import sys
import numpy as np
import pandas as pd
import importlib
import subprocess
import os
from dotenv import load_dotenv

plt.rcParams['figure.dpi'] = 120

import bombcell as bc

# ============= Load in configuration and helper functions =============
# Add MOUSE_NAME directory to path
mouse_dir = Path().resolve().parent.parent  # notebook in analyze_bc_results/
print("Notebook:", Path().resolve())
print("Added to sys.path:", mouse_dir)
print("Exists:", mouse_dir.exists())
print("Contents:", [p.name for p in mouse_dir.iterdir()])
sys.path.append(str(Path().resolve().parent))  # adds .../mice/Reach15

print("CWD:", Path().resolve())
print("sys.path[0:5]:", sys.path[0:5])
print("Has Reach15/grant_config.py?:", (Path().resolve().parent / "grant_config.py").exists())
print("Has Reach15/helper_func/grant_config.py?:", (Path().resolve().parent / "helper_func" / "grant_config.py").exists())

import helper_func.prep_data as data_prep
import helper_func.nwb_data_prep as prep
from helper_func.grant_config import load_grant_config
from helper_func.post_analysis_setup import load_post_analysis_context

# Always reload local modules so notebook uses latest patched code.
prep = importlib.reload(prep)
# plots = importlib.reload(plots)

print('pca_data_prep path:', Path(prep.__file__).resolve())
if not hasattr(prep, 'extract_probe_letters'):
    raise AttributeError(
        'Loaded pca_data_prep does not expose extract_probe_letters. '
        'Restart kernel and re-run this cell, then confirm it points to master/pca_data_prep.py.'
    )


✅ ipywidgets available - interactive GUI ready
Notebook: C:\Users\user\Documents\github\bombcell\mice\Reach15\run_bc
Added to sys.path: C:\Users\user\Documents\github\bombcell\mice
Exists: True
Contents: ['Reach15']
CWD: C:\Users\user\Documents\github\bombcell\mice\Reach15\run_bc
sys.path[0:5]: ['c:\\Users\\user\\anaconda3\\envs\\bombcell\\python311.zip', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\DLLs', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\Lib', 'c:\\Users\\user\\anaconda3\\envs\\bombcell', '']
Has Reach15/grant_config.py?: False
Has Reach15/helper_func/grant_config.py?: True
pca_data_prep path: C:\Users\user\Documents\github\bombcell\mice\Reach15\helper_func\nwb_data_prep.py


### Load data from .env

In [2]:
session_data_dic = prep.load_env()
session_data_dic

MOUSE loaded: Reach15
-- Behavioral Files --
BEHAVIORAL_FOLDER loaded: grant_reach15_swingDoor-christie
-- Neuropixels Sessions --
Session 1: NP_FILE=Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01 DATE=20260129 SESSION=session003 BOMBCELL=bombcell_batch_20260305_1130
Session 2: NP_FILE=Reach15_20260129_session004_NP_Recording_02_2026-01-29_16-50-32 DATE=20260129 SESSION=session004 BOMBCELL=NA
Session 3: NP_FILE=Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00 DATE=20260201 SESSION=session007 BOMBCELL=bombcell_batch_20260304_1536
-- Config Defaults --
RECORDINGS_ROOT loaded: H:/Grant/Neuropixels/Kilosort_Recordings
OPEN_EPHYS_CONTINUOUS_SUBPATH loaded: Record Node 103/experiment1/recording1/continuous
STRUCTURE_OEBIN_SUBPATH loaded: Record Node 103/experiment1/recording1/structure.oebin
NP20_PROBES loaded: A,C,D


{'MOUSE': 'Reach15',
 'BEHAVIORAL_FOLDER': 'grant_reach15_swingDoor-christie',
 'NP_FILE': 'Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01',
 'NWB_FILE': 'NA',
 'DATE': '20260129',
 'SESSION': 'session003',
 'BOMBCELL': 'bombcell_batch_20260305_1130',
 'NP_FILE_01': 'Reach15_20260129_session004_NP_Recording_02_2026-01-29_16-50-32',
 'NWB_FILE_01': 'NA',
 'DATE_01': '20260129',
 'SESSION_01': 'session004',
 'BOMBCELL_01': 'NA',
 'NP_FILE_02': 'Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00',
 'NWB_FILE_02': 'Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00',
 'DATE_02': '20260201',
 'SESSION_02': 'session007',
 'BOMBCELL_02': 'bombcell_batch_20260304_1536',
 'RECORDINGS_ROOT': 'H:/Grant/Neuropixels/Kilosort_Recordings',
 'OPEN_EPHYS_CONTINUOUS_SUBPATH': 'Record Node 103/experiment1/recording1/continuous',
 'STRUCTURE_OEBIN_SUBPATH': 'Record Node 103/experiment1/recording1/structure.oebin',
 'NP20_PROBES': 'A,C,D',
 'RESULTS_PATH'

#### =========================================================
## STEP 1: Select Neuropixel session to run bombcell on
#### =========================================================


In [3]:
SESSION_TO_ANALYZE = 1

MOUSE, BEHAVIORAL_FOLDER, NP_FILE, NWB_FILE, DATE, SESSION, BOMBCELL = prep.session_to_analyze(
                                                                                        session_data_dic['MOUSE'], session_data_dic['BEHAVIORAL_FOLDER'], 
                                                                                        session_data_dic['NP_FILE'],session_data_dic['NWB_FILE'] ,session_data_dic['DATE'], session_data_dic['SESSION'],session_data_dic['BOMBCELL'],
                                                                                        session_data_dic['NP_FILE_01'], session_data_dic['NWB_FILE_01'], session_data_dic['DATE_01'], session_data_dic['SESSION_01'],session_data_dic['BOMBCELL_01'] ,
                                                                                        session_data_dic['NP_FILE_02'], session_data_dic['NWB_FILE_02'], session_data_dic['DATE_02'], session_data_dic['SESSION_02'],session_data_dic['BOMBCELL_02'],
                                                                                        session_selection=SESSION_TO_ANALYZE
                                                                                    )
                                                               


SESSION SELECTION:

NP_FILE: Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01
NWB_FILE: NA
DATE: 20260129
SESSION: session003
BEHAVIORAL_FOLDER: grant_reach15_swingDoor-christie
BOMBCELL: bombcell_batch_20260305_1130


### Set and verify bombcell paths using loaded session data from .env

In [ ]:
BOMBCELL_ROOT_FOR_AUTO_BUILD, NWB_PATH, CONFIG_FILE, SESSION_NAME = data_prep.set_bc_paths(session_data_dic, MOUSE, NWB_FILE, NP_FILE, BOMBCELL, BEHAVIORAL_FOLDER, DATE, SESSION, SESSION_TO_ANALYZE)



AUTO-GENERATED SESSION CONFIG

Session selection: 1
Mouse root: C:\Users\user\Documents\github\bombcell\mice\Reach15
Recording name: Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01
Wrote config: C:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config_20260129_session003.json
All required files/folders found.
NWB file: H:\NWB_OUT\NA
Bombcell root folder: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130
Session config file: C:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config_20260129_session003.json


#### =======================================
## RUN BOMB CELL ANALYSIS
#### =======================================



#### Prepare the  cmd file

In [5]:
# Select run mode and target probe (if applicable)
RUN_MODE = 'single_probe'  # batch | single_probe | np20_rerun
TARGET_PROBE = 'A'  # only used for single_probe
OVERWRITE = True

# runner = Path('helper_func/run_bombcell_unified.py')
# cmd = ['python', str(runner), '--config', str(CONFIG_FILE), '--mode', RUN_MODE]

runner = Path("../helper_func/run_bombcell_unified.py").resolve()
cmd = ['python', str(runner), '--config', str(CONFIG_FILE), '--mode', RUN_MODE]


if RUN_MODE == 'single_probe':
    cmd += ['--target-probe', TARGET_PROBE]
if OVERWRITE:
    cmd.append('--overwrite')

print('Running:', ' '.join(cmd))
# subprocess.run(cmd, check=True)
cmd

Running: python C:\Users\user\Documents\github\bombcell\mice\Reach15\helper_func\run_bombcell_unified.py --config C:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config_20260129_session003.json --mode single_probe --target-probe A --overwrite


['python',
 'C:\\Users\\user\\Documents\\github\\bombcell\\mice\\Reach15\\helper_func\\run_bombcell_unified.py',
 '--config',
 'C:\\Users\\user\\Documents\\github\\bombcell\\mice\\Reach15\\configs\\grant_recording_config_20260129_session003.json',
 '--mode',
 'single_probe',
 '--target-probe',
 'A',
 '--overwrite']

### Start the Bombcell pipeline by running the above command in the terminal.

In [6]:
# new code
p = subprocess.run(cmd, text=True, capture_output=True)
print("Return code:", p.returncode)
print("\n--- STDOUT ---\n", p.stdout)
print("\n--- STDERR ---\n", p.stderr)
p.check_returncode()  # will raise after printing, if you still want it to


Return code: 0

--- STDOUT ---
 ✅ ipywidgets available - interactive GUI ready

[SINGLE_PROBE MODE] target_probe=A region=NP2.0 Simplex Lobule & Interposed Nucleus (SIM/IP)

Creating run root for mode 'single_probe' on 20260305 at 1743...

=== Probe A (NP2.0 Simplex Lobule & Interposed Nucleus (SIM/IP)) ===
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_single_probe_20260305_1743\kilosort4_A
raw_file: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\Record Node 103\experiment1\recording1\continuous\Neuropix-PXI-100.ProbeA\continuous.dat
meta_file: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\Record Node 103\experiment1\recording1\structure.oebin
save_path: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_single_probe_20260305_17

## View Results of BC run

#### Select BC file to analyze

In [39]:
bombcell_run = r'H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130'

In [40]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import bombcell as bc

analysis_dir = Path.cwd().resolve()
sys.path.insert(0, str(analysis_dir))
from post_analysis_setup import load_post_analysis_context

ctx = load_post_analysis_context(CONFIG_FILE)

staging_root = bombcell_run
print('staging_root:', staging_root)
probe_letters = list(ctx['probeLetters'])

for TARGET_PROBE in probe_letters:
    ks_dir = Path(staging_root) / f'kilosort4_{TARGET_PROBE}'
    save_path = ks_dir / 'bombcell'

    print('ks_dir:', ks_dir)
    print('save_path:', save_path)

    param, quality_metrics, _ = bc.load_bc_results(str(save_path))
    unit_type, unit_type_string = bc.qm.get_quality_unit_type(param, quality_metrics)
    qm_df = pd.DataFrame(quality_metrics).copy()
    qm_df['bombcell_label'] = unit_type_string
    qm_df['unit_index'] = np.arange(len(qm_df))

    if 'cluster_id' not in qm_df.columns:
        qm_df['cluster_id'] = qm_df['unit_index']

    print('Loaded units:', len(qm_df))
    print(qm_df['bombcell_label'].value_counts(dropna=False))

✅ ipywidgets available - interactive GUI ready
staging_root: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_A
save_path: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_A\bombcell
Loaded units: 961
bombcell_label
NOISE       492
MUA         268
NON-SOMA    200
GOOD          1
Name: count, dtype: int64
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_B
save_path: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_B\bombcell
Loaded units